# 02 Feature Engineering

Create rolling, windowed, and sequence features for RUL modeling.

In [1]:
from pathlib import Path
import pickle
import sys

import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features import (
    add_degradation_features,
    add_test_rul_labels,
    add_train_rul_labels,
    fit_transform_scaler,
    load_cmapss_subset,
    make_lstm_sequences,
    select_feature_columns,
)

DATA_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SUBSET = 'FD001'
RUL_CAP = 125
SEQ_LEN = 30
ROLLING_WINDOWS = (5, 10)

print(f'Using dataset directory: {DATA_DIR}')
print(f'Processed outputs will be saved to: {PROCESSED_DIR}')

Using dataset directory: C:\Users\Hamza\Desktop\Projects\rul-predictor\data\raw
Processed outputs will be saved to: C:\Users\Hamza\Desktop\Projects\rul-predictor\data\processed


## 1) Load and Label CMAPSS FD001
We load train/test/RUL files, then create capped RUL labels for both train and test frames.

In [ ]:
train_raw, test_raw, rul_raw = load_cmapss_subset(DATA_DIR, subset=SUBSET)
train_df = add_train_rul_labels(train_raw, cap=RUL_CAP)
test_df = add_test_rul_labels(test_raw, rul_raw, cap=RUL_CAP)

print(f'train_df shape: {train_df.shape}')
print(f'test_df shape : {test_df.shape}')
print('RUL stats (train):')
print(train_df['rul'].describe().round(2))

## 2) Sensor Selection and Feature Creation
We keep informative sensors, then add temporal features: first differences and rolling statistics.

In [ ]:
default_drop_sensors = ['s1', 's5', 's6', 's10', 's16', 's18', 's19']
eda_config_path = PROCESSED_DIR / 'eda_config.json'
if eda_config_path.exists():
    eda_config = pd.read_json(eda_config_path, typ='series').to_dict()
    drop_sensors = eda_config.get('drop_sensors', default_drop_sensors)
else:
    drop_sensors = default_drop_sensors

base_feature_cols = select_feature_columns(train_df, drop_sensors=drop_sensors)
sensor_cols = [c for c in base_feature_cols if c.startswith('s')]

train_feat = add_degradation_features(
    train_df,
    sensor_cols=sensor_cols,
    rolling_windows=ROLLING_WINDOWS,
    )
test_feat = add_degradation_features(
    test_df,
    sensor_cols=sensor_cols,
    rolling_windows=ROLLING_WINDOWS,
    )

feature_cols = [
    c
    for c in train_feat.columns
    if c not in {'engine_id', 'cycle', 'rul'}
    and pd.api.types.is_numeric_dtype(train_feat[c])
]

print(f'Dropped sensors: {drop_sensors}')
print(f'Base sensors kept: {len(sensor_cols)}')
print(f'Total model features after engineering: {len(feature_cols)}')
print(f'train_feat shape: {train_feat.shape}')
print(f'test_feat shape : {test_feat.shape}')

## 3) Scale Features and Save Tabular Artifacts
We fit a scaler on train features only, transform train/test, and save tabular files for baseline models.

In [ ]:
train_scaled, test_scaled, scaler = fit_transform_scaler(
    train_feat,
    test_feat,
    feature_cols=feature_cols,
    )

tabular_train_path = PROCESSED_DIR / 'train_fd001_features.parquet'
tabular_test_path = PROCESSED_DIR / 'test_fd001_features.parquet'
feature_list_path = PROCESSED_DIR / 'feature_columns_fd001.txt'
scaler_path = PROCESSED_DIR / 'fd001_scaler.pkl'

train_scaled.to_parquet(tabular_train_path, index=False)
test_scaled.to_parquet(tabular_test_path, index=False)
feature_list_path.write_text('\n'.join(feature_cols), encoding='utf-8')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f'Saved: {tabular_train_path.name}')
print(f'Saved: {tabular_test_path.name}')
print(f'Saved: {feature_list_path.name}')
print(f'Saved: {scaler_path.name}')

## 4) Build LSTM Sequences and Save Arrays
Create fixed-length windows from scaled features for sequence models.

In [ ]:
X_train, y_train, train_engines = make_lstm_sequences(
    train_scaled,
    feature_cols=feature_cols,
    seq_len=SEQ_LEN,
    target_col='rul',
    )
X_test, y_test, test_engines = make_lstm_sequences(
    test_scaled,
    feature_cols=feature_cols,
    seq_len=SEQ_LEN,
    target_col='rul',
    )

seq_out_path = PROCESSED_DIR / 'fd001_sequences.npz'
np.savez_compressed(
    seq_out_path,
    X_train=X_train,
    y_train=y_train,
    train_engines=train_engines,
    X_test=X_test,
    y_test=y_test,
    test_engines=test_engines,
    feature_cols=np.array(feature_cols),
    seq_len=np.array([SEQ_LEN]),
)

summary = {
    'subset': SUBSET,
    'rul_cap': RUL_CAP,
    'seq_len': SEQ_LEN,
    'rolling_windows': list(ROLLING_WINDOWS),
    'num_base_sensors': len(sensor_cols),
    'num_total_features': len(feature_cols),
    'train_rows': int(len(train_scaled)),
    'test_rows': int(len(test_scaled)),
    'X_train_shape': list(X_train.shape),
    'X_test_shape': list(X_test.shape),
}
summary_path = PROCESSED_DIR / 'feature_engineering_summary_fd001.json'
pd.Series(summary).to_json(summary_path, indent=2)

print(f'X_train shape: {X_train.shape}, y_train shape: {y_train.shape}')
print(f'X_test  shape: {X_test.shape}, y_test  shape: {y_test.shape}')
print(f'Saved: {seq_out_path.name}')
print(f'Saved: {summary_path.name}')